# Capital Bikeshare Demand Analytics & Machine Learning

**Python · pandas · scikit-learn · regression · classification · decision trees · ROC/AUC · cross-validation · Lasso**

## Project objective

This project analyzes 2025 Capital Bikeshare demand around the **22nd & H St NW** station near George Washington University. It combines four related supervised-learning coursework notebooks into one professional workflow:

1. Daily pickup/drop-off demand construction
2. Weather-data integration
3. Linear-regression benchmarking
4. KNN, logistic regression, and SVM classification
5. Decision-tree diagnostics, confusion matrix, and ROC/AUC
6. Lasso regularization and cross-validation
7. Business interpretation and limitations

> **Portfolio note:** Classroom task labels, point values, and personal Google Drive paths were removed.

> **Time-window note:** The source code includes clock hours 8, 9, and 10 (`dt.hour >= 8` and `<= 10`), approximately **8:00–10:59 AM**.


## Executive summary

The strongest source-reported simple linear-regression specification for both pickup and drop-off demand used **temperature + precipitation**.

A later decision-tree assignment adds an important evaluation lesson. The tree reached **85.62% test accuracy**, but the confusion matrix was `[[123, 2], [19, 2]]`. Because the test set contains 125 `DO_High` observations and only 21 `PU_High` observations, an always-`DO_High` majority classifier also reaches approximately **85.62% accuracy**.

Key source-reported results:

| Analysis | Result |
|---|---:|
| Pickup regression | Temp + Precip — Test MSE **4.0121** |
| Drop-off regression | Temp + Precip — Test MSE **20.8469** |
| Earlier classification block | KNN (k=6), Logistic Regression, RBF-SVC — **80.82%** test accuracy |
| Decision tree | **85.62%** test accuracy |
| Decision tree confusion matrix | `[[123, 2], [19, 2]]` |
| Linear SVM | Best later 5-fold CV AUC: **0.6144** |
| Selected Linear SVM | Final test AUC **0.5425** |
| Pickup Lasso | α **0.0494**, Test MSE **4.2814** |
| Drop-off Lasso | α **0.2683**, Test MSE **20.8275** |

**Split note:** the first three source notebooks use `random_state=2026`; the later decision-tree/AUC notebook uses `random_state=200`. The portfolio preserves both result blocks transparently rather than pretending they used the same test sample.


## 1. Setup and expected repository structure

Place the notebook inside `notebooks/` and raw data inside `data/raw/`.

```text
capital-bikeshare-demand-analytics/
├── README.md
├── notebooks/
│   └── capital_bikeshare_demand_analytics.ipynb
├── data/
│   ├── README.md
│   └── raw/
│       ├── ...capitalbikeshare-tripdata....csv
│       └── DC_weather_2025.csv
├── images/
├── results/
├── requirements.txt
└── .gitignore
```

Raw data are intentionally excluded from version control. See `data/README.md` for the required files and sources.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, Lasso, LassoCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import mean_squared_error, accuracy_score

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "raw"
RESULTS_DIR = ROOT / "results"

STATION = "22nd & H St NW"
START_DATE = "2025-01-01"
END_DATE = "2025-12-31"
RANDOM_STATE = 2026

print("Project root:", ROOT)
print("Data directory:", DATA_DIR)


## 2. Load Capital Bikeshare trip data

The original analysis combined all 2025 CSV files whose filenames contained `capitalbikeshare-tripdata`. The files are not committed to GitHub because the combined trip data are large.


In [ ]:
trip_files = sorted(DATA_DIR.glob("*capitalbikeshare-tripdata*.csv"))

if not trip_files:
    raise FileNotFoundError(
        "No Capital Bikeshare CSV files found in data/raw/. "
        "Download the 2025 trip files described in data/README.md."
    )

trips = pd.concat(
    (pd.read_csv(file, low_memory=False) for file in trip_files),
    ignore_index=True
)

trips["started_at"] = pd.to_datetime(trips["started_at"])
trips["ended_at"] = pd.to_datetime(trips["ended_at"])

print(f"Loaded {len(trip_files)} trip files.")
print(f"Rows: {len(trips):,}")
trips.head()


## 3. Create daily pickup and drop-off demand

Demand is defined at the selected station for each calendar day in 2025. Days with no observed trips are explicitly filled with zero.

- **PU_ct**: trips that start at the station during the morning window
- **DO_ct**: trips that end at the station during the morning window


In [ ]:
full_dates = pd.date_range(START_DATE, END_DATE, freq="D")

pickup_mask = (
    trips["start_station_name"].eq(STATION)
    & trips["started_at"].dt.hour.between(8, 10)
)
dropoff_mask = (
    trips["end_station_name"].eq(STATION)
    & trips["ended_at"].dt.hour.between(8, 10)
)

pickup_counts = (
    trips.loc[pickup_mask]
    .groupby(trips.loc[pickup_mask, "started_at"].dt.normalize())
    .size()
    .reindex(full_dates, fill_value=0)
)

dropoff_counts = (
    trips.loc[dropoff_mask]
    .groupby(trips.loc[dropoff_mask, "ended_at"].dt.normalize())
    .size()
    .reindex(full_dates, fill_value=0)
)

demand = pd.DataFrame({
    "date": full_dates,
    "PU_ct": pickup_counts.to_numpy(),
    "DO_ct": dropoff_counts.to_numpy(),
})

demand.head()


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(demand["date"], demand["PU_ct"], label="Pickups")
plt.plot(demand["date"], demand["DO_ct"], label="Drop-offs")
plt.xlabel("Date")
plt.ylabel("Morning trip count")
plt.title(f"Morning Pickups and Drop-offs at {STATION}")
plt.legend()
plt.tight_layout()
plt.show()


### Original executed visualization

![Morning pickup and drop-off demand](../images/demand_over_time.png)

The original analysis observed generally sparse morning station demand, with frequent low counts and occasional spikes. Pickups and drop-offs also varied through the year.


## 4. Load and merge weather data

The weather file used in the coursework was a 2025 Washington, DC export from Visual Crossing. The model features retained for this portfolio are:

- `temp`
- `precip`
- `windspeed`
- `uvindex`
- `icon`


In [ ]:
weather_file = DATA_DIR / "DC_weather_2025.csv"

if not weather_file.exists():
    raise FileNotFoundError(
        "DC_weather_2025.csv was not found in data/raw/. "
        "See data/README.md for instructions."
    )

weather = pd.read_csv(weather_file)
weather["datetime"] = pd.to_datetime(weather["datetime"])
weather["date"] = weather["datetime"].dt.normalize()

model_features = ["temp", "precip", "windspeed", "uvindex", "icon"]

missing = [col for col in model_features if col not in weather.columns]
if missing:
    raise ValueError(f"Weather file is missing required columns: {missing}")

merged = demand.merge(
    weather[["date"] + model_features],
    on="date",
    how="left"
)

print("Merged shape:", merged.shape)
print("Missing values in modeling columns:")
print(merged[["PU_ct", "DO_ct"] + model_features].isna().sum())
merged.head()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(merged["temp"], merged["PU_ct"], alpha=0.6, label="Pickup count")
ax.set_xlabel("Temperature")
ax.set_ylabel("Pickup count")
ax.set_title("Pickup Demand vs. Temperature")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Train/test split

The original coursework used a **60% training / 40% test** random split with `random_state=2026`. The same setup is retained here so the reported benchmark results remain comparable.


In [ ]:
X = merged[model_features].copy()
y = merged[["PU_ct", "DO_ct"]].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


## 6. Linear-regression feature progression

The analysis evaluates progressively richer feature sets. Weather `icon` is one-hot encoded only when it enters the model.


In [ ]:
feature_sets = {
    "Temp": ["temp"],
    "Temp + Precip": ["temp", "precip"],
    "Temp + Precip + Windspeed": ["temp", "precip", "windspeed"],
    "Temp + Precip + Windspeed + UV Index": ["temp", "precip", "windspeed", "uvindex"],
    "All Features (incl. Icon OHE)": ["temp", "precip", "windspeed", "uvindex", "icon"],
}

def prepare_design_matrices(X_train, X_test, columns):
    train_design = X_train[columns].copy()
    test_design = X_test[columns].copy()

    if "icon" in columns:
        train_design = pd.get_dummies(
            train_design, columns=["icon"], prefix="icon", drop_first=True
        )
        test_design = pd.get_dummies(
            test_design, columns=["icon"], prefix="icon", drop_first=True
        )
        test_design = test_design.reindex(
            columns=train_design.columns, fill_value=0
        )

    return train_design, test_design

def evaluate_linear_regression(target):
    rows = []
    for label, columns in feature_sets.items():
        Xtr, Xte = prepare_design_matrices(X_train, X_test, columns)
        model = LinearRegression()
        model.fit(Xtr, y_train[target])

        train_mse = mean_squared_error(y_train[target], model.predict(Xtr))
        test_mse = mean_squared_error(y_test[target], model.predict(Xte))

        rows.append({
            "Features": label,
            "Training MSE": train_mse,
            "Test MSE": test_mse,
        })
    return pd.DataFrame(rows)

pickup_lr_results = evaluate_linear_regression("PU_ct")
dropoff_lr_results = evaluate_linear_regression("DO_ct")

pickup_lr_results


In [ ]:
dropoff_lr_results


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(pickup_lr_results["Features"], pickup_lr_results["Training MSE"], marker="o", label="Training MSE")
plt.plot(pickup_lr_results["Features"], pickup_lr_results["Test MSE"], marker="o", label="Test MSE")
plt.ylabel("Mean Squared Error")
plt.xlabel("Feature set")
plt.title("Pickup Regression: Training vs. Test MSE")
plt.xticks(rotation=30, ha="right")
plt.legend()
plt.tight_layout()
plt.show()


### Reported regression results from the executed coursework

**Pickup demand**

| Features | Training MSE | Test MSE |
|---|---:|---:|
| Temp | 3.4046 | 4.0341 |
| Temp + Precip | 3.3724 | **4.0121** |
| + Windspeed | 3.3695 | 4.0398 |
| + UV Index | 3.2736 | 4.2460 |
| All features + icon OHE | **3.1761** | 4.3592 |

![Pickup regression MSE](../images/pickup_regression_mse.png)

**Drop-off demand**

| Features | Training MSE | Test MSE |
|---|---:|---:|
| Temp | 22.7032 | 21.2426 |
| Temp + Precip | 22.0235 | **20.8469** |
| + Windspeed | 21.7766 | 21.5028 |
| + UV Index | 21.7744 | 21.5148 |
| All features + icon OHE | **21.1318** | 21.4619 |

![Drop-off regression MSE](../images/dropoff_regression_mse.png)

### Interpretation

For both targets, **temperature + precipitation** produced the lowest test MSE among the linear-regression specifications. Training MSE generally fell as more variables were added, while test MSE did not. This is consistent with increasing model complexity without a corresponding gain in generalization.


## 7. Classification: should pickups exceed drop-offs?

A binary target is created:

\[
PU\_gt\_DO = 1 \quad \text{if } PU\_ct > DO\_ct
\]

This reframes the problem as a simple operational signal: whether the station may need relatively more bikes available than docks during the morning period.


In [ ]:
y_class_train = (y_train["PU_ct"] > y_train["DO_ct"]).astype(int)
y_class_test = (y_test["PU_ct"] > y_test["DO_ct"]).astype(int)

X_train_cls = pd.get_dummies(
    X_train, columns=["icon"], prefix="icon", drop_first=True
)
X_test_cls = pd.get_dummies(
    X_test, columns=["icon"], prefix="icon", drop_first=True
).reindex(columns=X_train_cls.columns, fill_value=0)

classification_rows = []

# KNN k=5
knn5 = KNeighborsClassifier(n_neighbors=5)
knn5.fit(X_train_cls, y_class_train)
classification_rows.append({
    "Model": "KNN (k=5)",
    "Training Accuracy": accuracy_score(y_class_train, knn5.predict(X_train_cls)),
    "Test Accuracy": accuracy_score(y_class_test, knn5.predict(X_test_cls)),
})

# Original coursework comparison of k=1..15
k_values = range(1, 16)
train_acc = []
test_acc = []
for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_cls, y_class_train)
    train_acc.append(accuracy_score(y_class_train, model.predict(X_train_cls)))
    test_acc.append(accuracy_score(y_class_test, model.predict(X_test_cls)))

best_k = list(k_values)[int(np.argmax(test_acc))]

# Logistic Regression
logit = LogisticRegression(random_state=RANDOM_STATE, solver="liblinear")
logit.fit(X_train_cls, y_class_train)
classification_rows.append({
    "Model": "Logistic Regression",
    "Training Accuracy": accuracy_score(y_class_train, logit.predict(X_train_cls)),
    "Test Accuracy": accuracy_score(y_class_test, logit.predict(X_test_cls)),
})

# Linear SVC - retained to reproduce coursework configuration
linear_svc = LinearSVC(C=10, random_state=RANDOM_STATE, dual=True, max_iter=2000)
linear_svc.fit(X_train_cls, y_class_train)
classification_rows.append({
    "Model": "Linear SVC (C=10)",
    "Training Accuracy": accuracy_score(y_class_train, linear_svc.predict(X_train_cls)),
    "Test Accuracy": accuracy_score(y_class_test, linear_svc.predict(X_test_cls)),
})

# RBF SVC
rbf_svc = SVC(C=10, kernel="rbf", random_state=RANDOM_STATE)
rbf_svc.fit(X_train_cls, y_class_train)
classification_rows.append({
    "Model": "RBF SVC (C=10)",
    "Training Accuracy": accuracy_score(y_class_train, rbf_svc.predict(X_train_cls)),
    "Test Accuracy": accuracy_score(y_class_test, rbf_svc.predict(X_test_cls)),
})

classification_results = pd.DataFrame(classification_rows)
print("Best k from the original plotted test-set comparison:", best_k)
classification_results


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(list(k_values), train_acc, marker="o", label="Training Accuracy")
plt.plot(list(k_values), test_acc, marker="o", label="Test Accuracy")
plt.xlabel("Number of neighbors (k)")
plt.ylabel("Accuracy")
plt.title("KNN Accuracy vs. k")
plt.xticks(list(k_values))
plt.legend()
plt.tight_layout()
plt.show()


### Reported classification results

| Model | Training Accuracy | Test Accuracy |
|---|---:|---:|
| KNN, k=5 | 86.30% | 79.45% |
| KNN, k=6 | — | **80.82%** |
| Logistic Regression | 86.30% | **80.82%** |
| Linear SVC, C=10 | 87.21% | 80.14% |
| RBF SVC, C=10 | 85.84% | **80.82%** |

The logistic-regression model assigned a probability of **0.0541** to `PU_ct > DO_ct` for the first test sample in the original run.

![KNN accuracy by k](../images/knn_accuracy_by_k.png)

> **Methodology note:** In the coursework, `k` was chosen by comparing accuracy on the test set. That reproduces the submitted analysis, but it is not the preferred production workflow because it uses the test set during model selection. A stronger next iteration would tune `k` using cross-validation on the training set and evaluate the selected model once on the untouched test set.


## 8. Decision tree and ROC/AUC diagnostic extension

The later supervised-learning assignment evaluates the same `PU_High` vs `DO_High` business target with a depth-3 classification tree and AUC-based model selection.

To preserve the reported outputs, this section uses that notebook's original **60/40 split with `random_state=200`**. It is therefore a separate diagnostic block from the earlier `random_state=2026` analysis.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

X_diag = merged[model_features].copy()
y_diag = merged[["PU_ct", "DO_ct"]].copy()

X_train_diag, X_test_diag, y_train_diag, y_test_diag = train_test_split(
    X_diag, y_diag, test_size=0.40, random_state=200
)

y_train_diag = y_train_diag.copy()
y_test_diag = y_test_diag.copy()

y_train_diag["Which_High"] = (
    y_train_diag["PU_ct"] > y_train_diag["DO_ct"]
).map({True: "PU_High", False: "DO_High"})

y_test_diag["Which_High"] = (
    y_test_diag["PU_ct"] > y_test_diag["DO_ct"]
).map({True: "PU_High", False: "DO_High"})

X_train_diag = pd.get_dummies(
    X_train_diag, columns=["icon"], prefix="icon", drop_first=True
)
X_test_diag = pd.get_dummies(
    X_test_diag, columns=["icon"], prefix="icon", drop_first=True
).reindex(columns=X_train_diag.columns, fill_value=0)

tree_model = DecisionTreeClassifier(
    max_depth=3, criterion="entropy", random_state=0
)
tree_model.fit(X_train_diag, y_train_diag["Which_High"])

tree_pred = tree_model.predict(X_test_diag)
print("Decision-tree accuracy:",
      accuracy_score(y_test_diag["Which_High"], tree_pred))


In [ ]:
plt.figure(figsize=(18, 8))
plot_tree(
    tree_model,
    feature_names=X_train_diag.columns.tolist(),
    class_names=tree_model.classes_.tolist(),
    filled=True,
    rounded=True,
)
plt.title("Decision Tree: PU_High vs DO_High")
plt.tight_layout()
plt.show()


In [ ]:
cm = confusion_matrix(
    y_test_diag["Which_High"],
    tree_pred,
    labels=tree_model.classes_,
)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=tree_model.classes_,
)
fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax)
plt.title("Decision Tree Confusion Matrix")
plt.tight_layout()
plt.show()

print(cm)


In [ ]:
# Match the source notebook's convention: DO_High is the positive class
pos_idx = list(tree_model.classes_).index("DO_High")
neg_idx = 1 - pos_idx

TP = cm[pos_idx, pos_idx]
FP = cm[neg_idx, pos_idx]
FN = cm[pos_idx, neg_idx]
TN = cm[neg_idx, neg_idx]

tpr = TP / (TP + FN)
fpr = FP / (FP + TN)

class_counts = y_test_diag["Which_High"].value_counts()
majority_accuracy = class_counts.max() / class_counts.sum()

print(f"TPR: {tpr:.4f}")
print(f"FPR: {fpr:.4f}")
print(f"Majority baseline accuracy: {majority_accuracy:.4f}")


In [ ]:
candidates = {
    "Linear SVM": SVC(kernel="linear", probability=True, random_state=0),
    "KNN (k=10)": KNeighborsClassifier(n_neighbors=10),
    "Decision Tree (depth=3)": DecisionTreeClassifier(
        max_depth=3, criterion="entropy", random_state=0
    ),
}

cv_auc = {}
for name, model in candidates.items():
    scores = cross_val_score(
        model,
        X_train_diag,
        y_train_diag["Which_High"],
        cv=5,
        scoring="roc_auc",
    )
    cv_auc[name] = scores.mean()
    print(f"{name}: {scores.mean():.4f}")

print("Selected:", max(cv_auc, key=cv_auc.get))


In [ ]:
best_model = SVC(kernel="linear", probability=True, random_state=0)
best_model.fit(X_train_diag, y_train_diag["Which_High"])

test_prob = best_model.predict_proba(X_test_diag)[:, 1]
final_auc = roc_auc_score(y_test_diag["Which_High"], test_prob)
print(f"Final test AUC: {final_auc:.4f}")


### Source-reported results

- Decision-tree test accuracy: **0.8562**
- Confusion matrix: **`[[123, 2], [19, 2]]`**
- `DO_High` TPR: **0.9840**
- `DO_High` FPR: **0.9048**
- Linear SVM 5-fold CV AUC: **0.6144**
- KNN (k=10) 5-fold CV AUC: **0.5140**
- Decision Tree (depth=3) 5-fold CV AUC: **0.5046**
- Selected Linear SVM test AUC: **0.5425**

![Decision tree](../images/decision_tree_depth3.png)

![Confusion matrix](../images/decision_tree_confusion_matrix.png)

### Why the headline accuracy is misleading

The confusion matrix contains **125 `DO_High`** observations and **21 `PU_High`** observations. An always-`DO_High` classifier therefore achieves:

**125 / 146 ≈ 85.62% accuracy**

That is effectively identical to the tree's 85.62% accuracy. The tree identifies only **2 of 21 `PU_High`** observations, showing why accuracy alone is not a sufficient metric for this imbalanced target.

The AUC analysis is therefore more informative about discrimination, and the final test AUC of **0.5425** indicates that the selected classifier still has substantial room for improvement.


## 9. Regularization and cross-validation with Lasso

The regularization analysis one-hot encodes `icon`, standardizes the design matrix, and evaluates Lasso regression over a range of α values.


In [ ]:
X_train_dummies = pd.get_dummies(
    X_train, columns=["icon"], prefix="icon", drop_first=True
)
X_test_dummies = pd.get_dummies(
    X_test, columns=["icon"], prefix="icon", drop_first=True
).reindex(columns=X_train_dummies.columns, fill_value=0)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_dummies),
    columns=X_train_dummies.columns,
    index=X_train_dummies.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_dummies),
    columns=X_test_dummies.columns,
    index=X_test_dummies.index,
)

alphas = np.logspace(-4, 2, 50)


In [ ]:
# 5-fold CV for Lasso(alpha=1), matching the coursework
kf = KFold(n_splits=5, shuffle=True, random_state=42)
alpha1_scores = cross_val_score(
    Lasso(alpha=1),
    X_train_scaled,
    y_train["PU_ct"],
    cv=kf,
    scoring="neg_mean_squared_error",
)

print("Average negative MSE, Lasso(alpha=1):", alpha1_scores.mean())

# GridSearchCV for pickup alpha
grid = GridSearchCV(
    Lasso(max_iter=10000),
    {"alpha": alphas},
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)
grid.fit(X_train_scaled, y_train["PU_ct"])

print("GridSearchCV best pickup alpha:", grid.best_params_["alpha"])
print("GridSearchCV best pickup negative MSE:", grid.best_score_)


In [ ]:
# LassoCV for pickup demand
lasso_cv_pu = LassoCV(
    alphas=alphas,
    cv=5,
    max_iter=10000,
    random_state=42,
    n_jobs=-1,
)
lasso_cv_pu.fit(X_train_scaled, y_train["PU_ct"])

final_lasso_pu = Lasso(alpha=lasso_cv_pu.alpha_, max_iter=10000)
final_lasso_pu.fit(X_train_scaled, y_train["PU_ct"])

pickup_lasso_test_mse = mean_squared_error(
    y_test["PU_ct"],
    final_lasso_pu.predict(X_test_scaled)
)

pickup_coefficients = pd.Series(
    final_lasso_pu.coef_,
    index=X_train_scaled.columns,
    name="coefficient"
).sort_values(key=np.abs, ascending=False)

print("Pickup best alpha:", lasso_cv_pu.alpha_)
print("Pickup Lasso test MSE:", pickup_lasso_test_mse)
pickup_coefficients


In [ ]:
# LassoCV for drop-off demand
lasso_cv_do = LassoCV(
    alphas=alphas,
    cv=5,
    max_iter=10000,
    random_state=42,
    n_jobs=-1,
)
lasso_cv_do.fit(X_train_scaled, y_train["DO_ct"])

final_lasso_do = Lasso(alpha=lasso_cv_do.alpha_, max_iter=10000)
final_lasso_do.fit(X_train_scaled, y_train["DO_ct"])

dropoff_lasso_test_mse = mean_squared_error(
    y_test["DO_ct"],
    final_lasso_do.predict(X_test_scaled)
)

print("Drop-off best alpha:", lasso_cv_do.alpha_)
print("Drop-off Lasso test MSE:", dropoff_lasso_test_mse)


### Reported regularization results

- Lasso(α=1), pickup 5-fold CV average negative MSE: **-3.9167**
- GridSearchCV best pickup α: **0.0494**
- GridSearchCV best pickup CV negative MSE: **-3.3905**
- LassoCV best pickup α: **0.0494**
- Pickup Lasso test MSE: **4.2814**
- LassoCV best drop-off α: **0.2683**
- Drop-off Lasso test MSE: **20.8275**

Pickup coefficients at the selected α:

| Feature | Coefficient |
|---|---:|
| temp | 0.4865 |
| precip | -0.0539 |
| windspeed | -0.0842 |
| uvindex | 0.3409 |
| icon_cloudy | -0.1668 |
| icon_partly-cloudy-day | -0.0017 |
| icon_rain | **0.0000** |
| icon_snow | 0.1313 |

The zero coefficient for `icon_rain` illustrates Lasso's ability to shrink a weak feature completely out of the fitted model.

![Lasso coefficient paths](../images/lasso_coefficient_paths.png)


## 10. Business interpretation

The results suggest that a relatively small set of weather variables can capture much of the predictive signal available in this dataset. For the simple linear-regression benchmarks, temperature and precipitation generalized better than larger feature sets.

From an operations perspective, a demand forecast could contribute to station rebalancing decisions, but this analysis should be treated as a **proof of concept**, not a deployment-ready optimization system. Reliable rebalancing would also require network-level station conditions, bike and dock availability, calendar effects, commuting patterns, events, and the cost of moving inventory.


## 11. Limitations and stronger next steps

This section is intentionally included because understanding model limitations is part of good analytics practice.

1. **Random split on time-indexed observations.** A future version should use a chronological holdout or rolling/expanding-window validation to better represent forecasting.
2. **KNN hyperparameter selection used the test set.** Tune hyperparameters only on the training set with cross-validation, then evaluate once on the test set.
3. **Scaling occurred before the Lasso CV step.** A production workflow should place `StandardScaler` and the estimator inside a `Pipeline` so preprocessing is refit separately within each CV fold.
4. **Sparse count outcomes.** Pickup and drop-off counts are non-negative and often small. Poisson/negative-binomial approaches, gradient boosting, or count-aware models may be worth comparing with ordinary least squares.
5. **Limited predictors.** Add day-of-week, month/season, holiday, school-calendar, major-event, lagged-demand, and nearby-station features.
6. **Station-level rather than network-level optimization.** A true rebalancing solution should model multiple stations jointly and incorporate operational costs/capacity constraints.

These improvements are natural extensions of the current proof of concept and would make the project substantially stronger for production or research use.


7. **Different source splits.** Earlier work uses `random_state=2026`; the later tree/AUC block uses `random_state=200`. Results are labeled separately.
8. **Class imbalance.** Future classification work should add balanced accuracy, precision/recall, PR-AUC, ROC-AUC, and class-aware validation.


## 12. Reproducibility and AI-assistance disclosure

The raw Capital Bikeshare and weather datasets are not redistributed in this repository; `data/README.md` documents the expected inputs.

**AI-assistance disclosure:** Generative AI tools were used during the original coursework to assist with workflow planning, code drafting/debugging, and refinement of written explanations. The reported model executions, outputs, and interpretations were reviewed by the author. This portfolio version reorganizes that work for reproducibility and professional presentation.


## 13. Key takeaway

The main lesson from this analysis is not that the largest model wins. In the linear-regression benchmark, **temperature + precipitation** provided the strongest test performance for both pickup and drop-off demand. Additional features improved training fit but did not improve generalization.

That result reinforces a practical analytics principle: **model complexity should be justified by out-of-sample performance, not by training fit alone.**
